<h1>What does a network see just before it decides?</h1>

We train a small MLP to classify the MNIST digits 0, 1 and 2. The layer before the classifier
has only **3 units**, so every image becomes 3 numbers (its *features*) that we can plot in 3D.

Fill in the cells marked `TODO`, look at the results, and write down what you notice.

<h1>Imports</h1>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

<h1>Data</h1>

In [ ]:
CLASSES = [0, 1, 2]
N = 5000  # samples per class (use ~800 for the test set)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

mnist = datasets.MNIST(root='./data', train=True, download=True)

x = torch.cat([mnist.data[mnist.targets == d][:N] for d in CLASSES])
x = (x.flatten(1).float() / 255 - 0.1307) / 0.3081    # [15000, 784], standard MNIST normalization
y = torch.arange(len(CLASSES)).repeat_interleave(N)   # [15000], integer labels 0/1/2
x, y = x.to(device), y.to(device)                     # whole dataset fits on the GPU

train_loader = DataLoader(TensorDataset(x, y), batch_size=64, shuffle=True)

<h1>Training a Model</h1>
<h3>Define a 5 layer MLP which can take in MNIST, and has hidden dimension of 3</h3>

Layer sizes: `784 → 400 → 100 → 20 → 3 → 3`, with a ReLU after every layer except the last.
The 3 outputs of `fc4` (after its ReLU) are the features we will study.

Create the layers in order `fc1` … `fc5` so the fixed seed below gives the same starting weights.

PyTorch references: [Build the Neural Network (tutorial)](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html) ·
[`nn.Module`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html) ·
[`nn.Linear`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html) ·
[`nn.ReLU`](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html)

In [ ]:
class MNIST_MLP(nn.Module):
  def __init__(self):
    super().__init__()
    self.relu = nn.ReLU()
    # TODO: define fc1 ... fc5

  def forward(self, x):
    # TODO: ReLU after fc1 ... fc4, nothing after fc5
    ...

<h3>Train the model on the subset dataset</h3>

PyTorch references: [Optimizing Model Parameters (tutorial)](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html) ·
[`DataLoader`](https://docs.pytorch.org/docs/stable/data.html) ·
[`F.mse_loss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.mse_loss.html) ·
[`F.one_hot`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html) ·
[`torch.optim.SGD`](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html) ·
[`torch.manual_seed`](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

In [ ]:
torch.manual_seed(16)  # fixed seed

model = MNIST_MLP().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

n_epochs = 200

for epoch in range(n_epochs):
    for inputs, labels in train_loader:
        # TODO: forward pass, MSE loss against the one-hot labels (hint: F.one_hot),
        #       then zero the gradients, backpropagate and step the optimizer
        ...

    print(f"epoch {epoch+1}: loss {loss.item():.4f}")

<h1>Feature Visualization</h1>

Use a forward hook on `fc4` to grab the 3 features of every image. A forward hook is a function
PyTorch calls every time a layer runs, with that layer's input and output.

PyTorch reference: [`register_forward_hook`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_forward_hook)

In [ ]:
# Hook that saves the penultimate features (fc4's output, after its ReLU)
activation = {}
def save_activation(layer, input, output):
    activation['penult'] = torch.relu(output.detach())

# TODO: register the hook on model.fc4, run every image through the model once, remove the hook

features = activation['penult'].cpu()   # [15000, 3]
labels = y.cpu()                         # [15000]

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt

K = 100  # points to plot per class

fig = plt.figure()
ax = fig.add_subplot(projection='3d')

for c, digit in enumerate(CLASSES):
    points = features[labels == c][:K]
    ax.scatter(points[:, 0], points[:, 1], points[:, 2], label=f'digit {digit}')

ax.set_xlabel('unit 0')
ax.set_ylabel('unit 1')
ax.set_zlabel('unit 2')
ax.legend()

plt.show()

**What do you notice?** Rotate the plot. Write down at least three observations before you go on.

1.
2.
3.

<h1>Next steps</h1>

For each one, predict what will happen before you run it.

1. **Watch it happen.** Save the features at a few epochs (e.g. 0, 1, 5, 20, 100) and plot them
   side by side. When does the structure appear?
2. **How tight are the clouds?** Find a number that compares the spread inside each class with
   the distance between classes. How does it change over training?
3. **Is it the same across digits?** Try different digits than 0, 1, and 2 to see if the structure still emerges
4. **Change the training:** [Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)
   instead of SGD, `weight_decay=0`, or
   [cross-entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)
   instead of MSE. Which of your observations still hold?
5. **More classes.** Use 4 digits, still with 3 features. What shape should the class means make?
   What about 5?